# Paper figures — DWI denoising

This notebook produces the paper figures for the supervised DWI-denoising model.

**Part 1 — augmentation effects.** For one held-out subject and slice it shows how
each on-the-fly augmentation (Rician noise, k-space truncation, b-vector / direction
masking) and their combination corrupts a *classical* (naive WLS) DTI reconstruction
of FA and ADC.

**Part 2 — model performance.** It runs the production
`runs/production_6d_tiny_rician_dropped` model (**AI**) together with the two
exported ONNX graphs in `onnx_models/` (**MRD-CNN**, **QSpace-Attention**) on the
held-out test subjects across increasing degradation levels and compares, for FA
and ADC, every method side by side — **Target**, the **Naive** WLS fit, the
classical denoisers **BM4D / Patch2Self / MP-PCA**, and the three neural models.
A summary plot quantifies reconstruction error vs degradation level.

**Part 3 — per-voxel agreement.** Density-scatter and Bland-Altman panels pool all
brain voxels at one degradation level for every method, both ONNX models included.

All displayed maps are restricted to the DIPY brain mask (non-brain voxels are
black). The axial slice is set by `SLICE_IDX` in the setup cell. Run top-to-bottom
with the project's `.venv` kernel (it has `zarr`, `torch`, `dipy`).


In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib
import matplotlib as mpl
try:
    get_ipython  # noqa: F821  (defined only inside IPython/Jupyter)
except NameError:
    matplotlib.use("Agg")  # headless when run as a plain script
import matplotlib.pyplot as plt

# Put the project root and src/ on the path (mirrors the root entry points).
REPO = Path.cwd()
while not (REPO / "config.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
for p in (str(REPO / "src"), str(REPO)):
    if p not in sys.path:
        sys.path.insert(0, p)

import config as cfg
import zarr
from dw_thi.runtime import get_device
from dw_thi.augment import degrade_dwi_volume
from dw_thi.model import QSpaceUNet
from dw_thi.evaluate import predict_subject
from dw_thi.utils import (
    fit_dti_to_6d, dti6d_to_scalar_maps, sanitize_dti6d,
    scalar_map_metrics, select_plot_indices,
    _symmetric_limits, _robust_limits,
)

device = get_device()

ZARR = str(REPO / "dataset" / "default_clean.zarr")
RUN = REPO / "runs" / "production_6d_tiny_rician_dropped"
CKPT = str(RUN / "best_model.pt")
TEST_SUBJECTS = ["sub-03_ses-1", "sub-04_ses-1"]   # from the run's config.json
FIG_DIR = RUN / "paper_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

B0 = cfg.B0_THRESHOLD
FIT = cfg.DTI_FIT_METHOD          # "WLS"
SEED = cfg.EVAL_DEGRADE_SEED      # fixed -> reproducible degradation
MAXD = cfg.MAX_DIFFUSIVITY        # eigenvalue cap used by sanitize_dti6d

# Axial slice (z) to plot. None -> auto-select the slice with the most signal.
# An int picks that slice; negatives index from the end, so SLICE_IDX = -1 is
# the last (top) slice of the acquired slab.
SLICE_IDX = -1

# Combined degradation levels driving both the part-1 "combined" figure and the
# part-2 model sweep. keep_dirs = fraction of diffusion-weighted directions
# (b-vectors) retained — this is the AUG_BVEC_MASK training augmentation that
# simulates scanning fewer directions. b0 volumes are always kept.
LEVELS = [
    {"name": "mild",     "keep_fraction": 0.70, "noise": 0.05, "keep_dirs": 0.80},
    {"name": "moderate", "keep_fraction": 0.60, "noise": 0.15, "keep_dirs": 0.50},
    {"name": "severe",   "keep_fraction": 0.50, "noise": 0.25, "keep_dirs": 0.15},
]

# Colormaps with a black "bad" colour so masked-out (NaN) background is black.
def _cmap(name, bad="black"):
    c = mpl.colormaps[name].copy()
    c.set_bad(bad)
    return c

FA_CM, ADC_CM, DIFF_CM = _cmap("viridis"), _cmap("magma"), _cmap("bwr")

# Pin colours explicitly so the figures look the same regardless of the active
# matplotlib style or a dark JupyterLab theme. (A dark theme otherwise makes the
# inline backend draw text in a light colour, which then vanishes on our white
# canvas — the "missing text" symptom.)
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 13, "axes.titlesize": 13, "figure.titlesize": 16,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "savefig.facecolor": "white", "savefig.edgecolor": "white",
    "savefig.transparent": False,
    "text.color": "black", "axes.labelcolor": "black", "axes.titlecolor": "black",
    "xtick.color": "black", "ytick.color": "black", "axes.edgecolor": "black",
})

# Render inline only when actually in a notebook; never call plt.show() (avoids
# the "FigureCanvasAgg is non-interactive" warning under headless/Agg backends).
try:
    _IPY = get_ipython()  # noqa: F821
except NameError:
    _IPY = None


def show_close(fig, out):
    fig.savefig(out)
    if _IPY is not None:
        from IPython.display import Image, display
        display(Image(filename=str(out)))
    plt.close(fig)
    print("wrote", out)


print("device:", device, "| repo:", REPO)
print("figures ->", FIG_DIR, "| SLICE_IDX:", SLICE_IDX)


## Helpers

Thin orchestration over the existing `dw_thi` helpers — nothing reimplemented.
`fa_adc` sanitizes the tensor (PSD projection + eigenvalue cap) before deriving
FA/ADC, matching `evaluate._compute_dti_metrics`. `mask_directions` reproduces the
training b-vector masking (`AUG_BVEC_MASK`). `brain_only` blanks non-brain voxels
to NaN so only the masked brain is rendered. `pick_slice` resolves `SLICE_IDX`
(auto or an explicit, possibly negative, index).


In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────────
def load_subject(key):
    g = zarr.open_group(ZARR, mode="r")[key]
    return {
        "dwi":   np.asarray(g["target_dwi"][:], np.float32),     # (X, Y, Z, N)
        "dti6d": np.asarray(g["target_dti_6d"][:], np.float32),  # (X, Y, Z, 6)
        "bvals": np.asarray(g["bvals"][:], np.float32),          # (N,)
        "bvecs": np.asarray(g["bvecs"][:], np.float32),          # (3, N)
        "mask":  np.asarray(g["brain_mask"][:], bool),           # (X, Y, Z)
    }


def pick_slice(dwi, bvals):
    """Resolve SLICE_IDX to a non-negative axial index. None -> auto-select."""
    nz = dwi.shape[2]
    if SLICE_IDX is None:
        return int(select_plot_indices(dwi, bvals, B0)[0])
    return int(SLICE_IDX % nz)


def fa_adc(dti6d):
    """Sanitize then return (FA, ADC) maps."""
    return dti6d_to_scalar_maps(sanitize_dti6d(dti6d, MAXD))


def brain_only(img2d, mask2d):
    """Copy of a 2D map with non-brain voxels set to NaN (rendered as black)."""
    out = np.array(img2d, dtype=np.float32)
    out[~mask2d] = np.nan
    return out


def mask_directions(bvals, keep_frac, seed=0):
    """Boolean (N,) marking diffusion-weighted volumes (b-vectors) that are
    *dropped* so only `keep_frac` of the DW directions remain — the
    AUG_BVEC_MASK training augmentation, which simulates acquiring fewer
    gradient directions. b0 volumes are always kept; the same set of directions
    is dropped for every voxel (whole-direction masking, matching training)."""
    bvals = np.asarray(bvals)
    N = bvals.shape[0]
    dw_idx = np.where(bvals >= B0)[0]
    dropped = np.zeros(N, dtype=bool)
    if keep_frac < 1.0 and dw_idx.size:
        rng = np.random.default_rng(seed + 100_000)
        n_keep = max(1, int(round(dw_idx.size * keep_frac)))
        keep_set = rng.choice(dw_idx, size=n_keep, replace=False)
        dropped[dw_idx] = True
        dropped[keep_set] = False
    return dropped


def apply_masking(dwi, dropped):
    out = dwi.copy()
    out[..., dropped] = 0.0
    return out


def degrade(dwi, keep_fraction=1.0, noise=0.0, dropped=None, seed=SEED):
    """k-space cutout + Rician noise (+ optional b-vector masking). keep_fraction=1
    and noise=0 is a near-identity (only the Nyquist line is touched)."""
    out = dwi
    if keep_fraction < 1.0 or noise > 0.0:
        out = degrade_dwi_volume(
            out, keep_fraction=keep_fraction, rel_noise_level=noise, seed=seed,
            noise_distribution=cfg.NOISE_DISTRIBUTION, n_coils=cfg.NOISE_COILS,
        )
    if dropped is not None:
        out = apply_masking(out, dropped)
    return out


def naive_dti(dwi, bvals, bvecs, kept=None):
    """Plain WLS DTI fit on degraded DWI (the 'normal' reconstruction). When
    directions are masked, fit only the kept volumes so zeros don't corrupt it."""
    bvecs_n3 = bvecs.T if bvecs.shape[0] == 3 else bvecs
    if kept is not None:
        dwi, bvals, bvecs_n3 = dwi[..., kept], bvals[kept], bvecs_n3[kept]
    return fit_dti_to_6d(
        np.maximum(dwi, 0.0), bvals, bvecs_n3=bvecs_n3,
        fit_method=FIT, b0_threshold=B0,
    )


def rmse(ref2d, est2d, mask2d):
    return scalar_map_metrics(ref2d, est2d, mask=mask2d)["rmse"]


## Part 1 — Effect of the augmentations on classical FA/ADC

Each figure: rows = FA (viridis) / ADC (magma); columns = clean target followed
by a mini-sweep of increasing augmentation strength. Each degraded column is a
naive WLS DTI fit; titles report the brain-masked RMSE of that slice vs the
clean target. Maps are restricted to the brain mask.


In [ ]:
# ── Part 1: per-augmentation mini-sweeps ─────────────────────────────────────
P1_SUBJ = "sub-03_ses-1"
s1 = load_subject(P1_SUBJ)
z1 = pick_slice(s1["dwi"], s1["bvals"])
brain1 = s1["mask"][:, :, z1]
sub_vol = s1["dwi"][:, :, z1:z1 + 1, :]                      # single-slice volume
_tfa, _tadc = fa_adc(s1["dti6d"][:, :, z1:z1 + 1, :])
tgt_fa1, tgt_adc1 = _tfa[:, :, 0], _tadc[:, :, 0]
adc_lim1 = _robust_limits(tgt_adc1[brain1])                 # shared ADC scale (brain)


def p1_column(label, keep_fraction=1.0, noise=0.0, keep_dirs=1.0):
    dropped = (mask_directions(s1["bvals"], keep_dirs, seed=SEED)
               if keep_dirs < 1.0 else None)
    kept = (~dropped) if dropped is not None else None
    deg = degrade(sub_vol, keep_fraction, noise, dropped, seed=SEED)
    fa, adc = fa_adc(naive_dti(deg, s1["bvals"], s1["bvecs"], kept))
    fa, adc = fa[:, :, 0], adc[:, :, 0]
    note = f"FA {rmse(tgt_fa1, fa, brain1):.3f}   ADC {rmse(tgt_adc1, adc, brain1):.1e}"
    return {"label": label, "fa": fa, "adc": adc, "note": note}


def render_grid(title, panels, out, adc_lim, mask2d):
    n = len(panels)
    fig, axes = plt.subplots(2, n, figsize=(3.5 * n, 7.2), squeeze=False)
    for j, p in enumerate(panels):
        ttl = p["label"] + ("\n" + p["note"] if p.get("note") else "")
        axes[0, j].imshow(np.rot90(brain_only(p["fa"], mask2d)), cmap=FA_CM, vmin=0, vmax=1)
        axes[0, j].set_title(ttl, fontsize=13, color="black")
        axes[0, j].axis("off")
        axes[1, j].imshow(np.rot90(brain_only(p["adc"], mask2d)), cmap=ADC_CM,
                          vmin=adc_lim[0], vmax=adc_lim[1])
        axes[1, j].axis("off")
    fig.text(0.012, 0.71, "FA", rotation=90, va="center", fontsize=16,
             weight="bold", color="black")
    fig.text(0.012, 0.26, "ADC", rotation=90, va="center", fontsize=16,
             weight="bold", color="black")
    fig.suptitle(title, fontsize=15, y=0.99, color="black")
    fig.tight_layout(rect=(0.045, 0, 1, 0.92))
    show_close(fig, out)


clean_panel = {"label": "Clean (target)", "fa": tgt_fa1, "adc": tgt_adc1, "note": ""}

# Every per-augmentation sweep below isolates ONE knob and steps it through the
# values defined in LEVELS (mild -> moderate -> severe), so the single-knob
# figures stay in lock-step with the combined sweep and the Part-2 model
# evaluation. The level name is shown alongside the knob value in each title.
render_grid(
    f"Effect of Rician noise on classical FA/ADC  ({P1_SUBJ}, z={z1})",
    [clean_panel] + [p1_column(f"{L['name']}\nnoise={L['noise']}", noise=L["noise"])
                     for L in LEVELS],
    FIG_DIR / "fig_aug_rician.png", adc_lim1, brain1,
)
render_grid(
    f"Effect of k-space truncation on classical FA/ADC  ({P1_SUBJ}, z={z1})",
    [clean_panel] + [p1_column(f"{L['name']}\nkeep={L['keep_fraction']}",
                               keep_fraction=L["keep_fraction"]) for L in LEVELS],
    FIG_DIR / "fig_aug_kspace.png", adc_lim1, brain1,
)
render_grid(
    f"Effect of b-vector masking (fewer directions) on classical FA/ADC  ({P1_SUBJ}, z={z1})",
    [clean_panel] + [p1_column(f"{L['name']}\n{int(L['keep_dirs'] * 100)}% dirs",
                               keep_dirs=L["keep_dirs"]) for L in LEVELS],
    FIG_DIR / "fig_aug_masking.png", adc_lim1, brain1,
)
render_grid(
    f"Combined degradation on classical FA/ADC  ({P1_SUBJ}, z={z1})",
    [clean_panel] + [p1_column(L["name"], keep_fraction=L["keep_fraction"],
                               noise=L["noise"], keep_dirs=L["keep_dirs"]) for L in LEVELS],
    FIG_DIR / "fig_aug_combined.png", adc_lim1, brain1,
)


## Part 2 — Load the trained model + exported ONNX models

The production `QSpaceUNet` checkpoint is loaded as the **AI** method. The two
exported ONNX graphs in `onnx_models/` — **MRD-CNN** and **QSpace-Attention** —
are then loaded through an ONNX Runtime session and wrapped so they expose the
same `forward(signal, bvals, bvecs, vol_mask)` interface as `QSpaceUNet`, making
them drop-in models for `predict_subject`. All three neural methods therefore run
through the identical preprocessing / unscaling pipeline as every other method.

In [ ]:
# ── Load checkpoint -> QSpaceUNet ────────────────────────────────────────────
ckpt = torch.load(CKPT, map_location=device, weights_only=False)
model = QSpaceUNet(
    max_n=ckpt["max_n"],
    feat_dim=ckpt.get("feat_dim", 64),
    channels=tuple(ckpt.get("channels", [64, 128, 256, 512])),
    cholesky=ckpt.get("cholesky", False),
).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
DTI_SCALE = ckpt.get("dti_scale", 1.0)
MAX_BVAL = ckpt.get("max_bval", 1000.0)
print(f"epoch {ckpt.get('epoch')} | dti_scale={DTI_SCALE:.2f} | "
      f"max_bval={MAX_BVAL} | cholesky={ckpt.get('cholesky')}")


In [ ]:
# ── Load exported ONNX models (ONNX Runtime) ─────────────────────────────────
# The two graphs in onnx_models/ were exported from separate research models
# (MRD-CNN and an attention q-space net). Both share the QSpaceUNet I/O
# signature — forward(signal, bvals, bvecs, vol_mask) -> (B, 6, H, W) — but with
# FIXED canonical spatial dims (132x132) and N (max_n=258). OnnxQSpaceModel wraps
# an ORT session so it looks exactly like the torch model to predict_subject: it
# zero-pads each slice bottom-right to canonical_hw (mirroring dataset.py), runs
# the session, then crops the prediction back to the native size. The Cholesky ->
# PSD conversion and DTI scaling are already baked into the graph, so
# predict_subject's b0-normalisation, max_bval scaling and 1/dti_scale unscaling
# apply unchanged. Each model carries its own dti_scale / max_bval from its
# sidecar metadata.json.
import json

import onnxruntime as ort


class OnnxQSpaceModel:
    """ONNX-Runtime drop-in for QSpaceUNet (see dw_thi.evaluate.predict_subject).

    predict_subject only needs `.max_n`, `.eval()` and a callable taking
    (signal, bvals, bvecs, vol_mask) torch tensors and returning (B, 6, H, W).
    """

    def __init__(self, onnx_path, max_n, canonical_hw, providers=None):
        self.session = ort.InferenceSession(
            str(onnx_path), providers=providers or ["CPUExecutionProvider"],
        )
        self.max_n = int(max_n)
        self.canonical_hw = tuple(canonical_hw)
        self._out_names = [o.name for o in self.session.get_outputs()]

    def eval(self):                       # predict_subject calls model.eval()
        return self

    def __call__(self, *args):
        return self.forward(*args)

    def forward(self, signal, bvals, bvecs, vol_mask):
        # signal (B, N, H, W): pad H/W bottom-right to canonical, run, crop back.
        _, _, h, w = signal.shape
        ch, cw = self.canonical_hw
        ph, pw = ch - h, cw - w
        sig = torch.nn.functional.pad(signal, (0, pw, 0, ph)) if (ph or pw) else signal
        feeds = {
            "signal":   np.ascontiguousarray(sig.detach().cpu().numpy(), np.float32),
            "bvals":    np.ascontiguousarray(bvals.detach().cpu().numpy(), np.float32),
            "bvecs":    np.ascontiguousarray(bvecs.detach().cpu().numpy(), np.float32),
            "vol_mask": np.ascontiguousarray(vol_mask.detach().cpu().numpy(), np.float32),
        }
        out = self.session.run(self._out_names, feeds)[0]   # (B, 6, ch, cw)
        return torch.from_numpy(out[:, :, :h, :w]).to(signal.device)


def load_onnx_model(name):
    """Load onnx_models/<name>.onnx plus its metadata sidecar."""
    onnx_dir = REPO / "onnx_models"
    meta = json.loads((onnx_dir / f"{name}.onnx.metadata.json").read_text())
    mdl = OnnxQSpaceModel(onnx_dir / f"{name}.onnx", meta["max_n"], meta["canonical_hw"])
    print(f"loaded ONNX {name:18} | epoch {meta.get('epoch')} | "
          f"dti_scale={meta['dti_scale']:.2f} | max_bval={meta['max_bval']} | "
          f"cholesky={meta.get('cholesky')} | providers={mdl.session.get_providers()}")
    return mdl, meta


_mrd_model, _mrd_meta = load_onnx_model("mrd_cnn")
_attn_model, _attn_meta = load_onnx_model("qspace_attention")

# Registry of neural models driven by predict_subject. Each entry carries the
# model plus its own dti_scale / max_bval so unscaling matches how it was trained.
# Classical denoisers (naive/bm4d/patch2self/mppca) are NOT here — they go through
# `reconstruct` instead.
MODEL_REGISTRY = {
    "ai":               {"model": model,       "dti_scale": DTI_SCALE,                "max_bval": MAX_BVAL},
    "mrd_cnn":          {"model": _mrd_model,  "dti_scale": _mrd_meta["dti_scale"],   "max_bval": _mrd_meta["max_bval"]},
    "qspace_attention": {"model": _attn_model, "dti_scale": _attn_meta["dti_scale"],  "max_bval": _attn_meta["max_bval"]},
}
print("model-based methods:", list(MODEL_REGISTRY))


## Part 2 — Reconstruction performance across degradation levels

For each test subject and degradation level the same degraded DWI is reconstructed
by every method — **Naive** WLS fit, the classical denoisers **BM4D**,
**Patch2Self** and **MP-PCA** (denoise → WLS fit), and the three neural models
**AI** (production QSpaceUNet), **MRD-CNN** and **QSpace-Attention** (the two
exported ONNX graphs) — and shown side by side for FA (top) and ADC (bottom),
restricted to the brain mask. Brain-masked FA/ADC RMSE is collected for every
method for the summary. (BM4D is skipped if the `bm4d` package is not installed.)

In [ ]:
# ── Denoising baselines + reconstruction registry ────────────────────────────
# BM4D, Patch2Self and MP-PCA are classical denoisers: they clean the degraded
# DWI, after which the *same* naive WLS DTI fit (`naive_dti`) turns the denoised
# signal into a 6D tensor. Together with the raw "naive" fit and the neural
# models (the torch "ai" net and the two ONNX graphs, MRD-CNN and
# QSpace-Attention) they form one comparison set, all driven from the identical
# degraded volume / kept-direction mask produced in Part 2 — so every method is
# evaluated on exactly the same data.
import time

from dw_thi.evaluate import _run_patch2self, _run_mppca  # reuse the eval-time denoisers

try:
    import bm4d as _bm4d  # noqa: F401
    HAVE_BM4D = True
except Exception as _bm4d_err:                      # pragma: no cover
    HAVE_BM4D = False
    print(f"bm4d not importable -> BM4D baseline skipped ({_bm4d_err}). "
          f"`pip install bm4d` to enable it.")

# sigma=None -> estimate the noise level per DWI volume via the MAD of that
# volume (robust std). Set a float to force a fixed sigma. profile is the BM4D
# quality profile ("np" = normal).
BM4D_CFG = {"sigma": None, "profile": "np"}


def _run_bm4d(noisy: np.ndarray) -> np.ndarray:
    """Apply BM4D independently to each 3D DWI volume.

    Sigma is either taken from BM4D_CFG or estimated per-volume via MAD.
    """
    from bm4d import bm4d
    N = noisy.shape[3]
    denoised = np.empty_like(noisy, dtype=np.float32)
    sigma_cfg = BM4D_CFG["sigma"]
    profile = BM4D_CFG["profile"]
    t_start = time.time()
    for n in range(N):
        vol = noisy[:, :, :, n].astype(np.float64)
        if sigma_cfg is not None:
            sigma = float(sigma_cfg)
        else:
            sigma = float(np.median(np.abs(vol - np.median(vol))) / 0.6745)
            sigma = max(sigma, 1e-6)
        print(f"  BM4D  [{n + 1:2d}/{N}]  sigma={sigma:.4f} ...", end="", flush=True)
        t0 = time.time()
        denoised[:, :, :, n] = bm4d(vol, sigma, profile=profile).astype(np.float32)
        vol_elapsed = time.time() - t0
        total_elapsed = time.time() - t_start
        remaining = (total_elapsed / (n + 1)) * (N - n - 1)
        print(f"  {vol_elapsed:.1f}s  (remaining ~{remaining:.0f}s)", flush=True)
    return denoised


def _kept_subset(deg, bvals, bvecs, kept):
    """Clamp to >=0 and, when directions are masked, drop the zeroed volumes so
    the denoisers/fit only see real signal (mirrors `naive_dti`)."""
    bvecs_n3 = bvecs.T if bvecs.shape[0] == 3 else bvecs
    if kept is not None:
        deg, bvals, bvecs_n3 = deg[..., kept], bvals[kept], bvecs_n3[kept]
    return np.maximum(deg, 0.0).astype(np.float32), bvals, bvecs_n3


def reconstruct(method, deg, bvals, bvecs, kept=None, mask=None):
    """Reconstruct a 6D DTI tensor from one degraded DWI volume via `method`
    in {"naive", "bm4d", "patch2self", "mppca"}. Each denoiser cleans the DWI
    first; all share the same WLS fit, b0 threshold and kept directions."""
    dwi, bv, bvec_n3 = _kept_subset(deg, bvals, bvecs, kept)
    if method == "naive":
        clean = dwi
    elif method == "bm4d":
        clean = _run_bm4d(dwi)
    elif method == "patch2self":
        clean = _run_patch2self(dwi, bv)
    elif method == "mppca":
        clean = _run_mppca(dwi, mask=mask)
    else:
        raise ValueError(f"unknown reconstruction method: {method!r}")
    return fit_dti_to_6d(np.maximum(clean, 0.0), bv, bvecs_n3=bvec_n3,
                         fit_method=FIT, b0_threshold=B0)


# Display order + plotting colour for every method. BM4D is dropped from the
# active set when the package is unavailable so the notebook still runs.
METHODS = [
    ("naive",            "Naive recon",        "#9E9E9E"),
    ("bm4d",             "BM4D",               "#9C27B0"),
    ("patch2self",       "Patch2Self",         "#FF9800"),
    ("mppca",            "MP-PCA",             "#4CAF50"),
    ("ai",               "AI prediction",      "#2196F3"),
    ("mrd_cnn",          "MRD-CNN (ONNX)",     "#E91E63"),
    ("qspace_attention", "QSpace-Attn (ONNX)", "#00ACC1"),
]
ACTIVE_METHODS = [m for m in METHODS if m[0] != "bm4d" or HAVE_BM4D]
METHOD_LABEL = {k: lbl for k, lbl, _c in METHODS}
METHOD_COLOR = {k: c for k, _l, c in METHODS}
print("active reconstruction methods:", [k for k, _l, _c in ACTIVE_METHODS])


In [ ]:
# ── Part 2: run every method + render comparison figures ─────────────────────
def predict_model(key, subj, deg, kept):
    """Run one neural model from MODEL_REGISTRY through predict_subject with its
    own dti_scale / max_bval (covers the torch 'ai' net and both ONNX models)."""
    spec = MODEL_REGISTRY[key]
    return predict_subject(
        spec["model"], ZARR, subj, device,
        b0_threshold=B0, dti_scale=spec["dti_scale"], max_bval=spec["max_bval"],
        input_dwi=deg, vol_mask=kept.astype(np.float32),
        batch_size=cfg.EVAL_INFER_BATCH_SIZE,
    )


def recon_method(key, subj, deg, bvals, bvecs, kept, mask):
    """DTI-6D for one method on the *same* degraded input. Neural models in
    MODEL_REGISTRY ('ai', 'mrd_cnn', 'qspace_attention') see all volumes via
    vol_mask; every classical method is the denoise->WLS-fit path in
    `reconstruct` (sees the kept subset)."""
    if key in MODEL_REGISTRY:
        return predict_model(key, subj, deg, kept)
    return reconstruct(key, deg, bvals, bvecs, kept=kept, mask=mask)


def render_methods(subj, level, z, brain2d, tgt_maps, method_maps, out):
    """Rows = FA / ADC; columns = Target followed by every active method."""
    labels = ["Target"] + [METHOD_LABEL[k] for k, _l, _c in ACTIVE_METHODS]
    fa_imgs = [tgt_maps[0]] + [method_maps[k][0] for k, _l, _c in ACTIVE_METHODS]
    adc_imgs = [tgt_maps[1]] + [method_maps[k][1] for k, _l, _c in ACTIVE_METHODS]
    n = len(labels)
    adc_lim = _robust_limits(tgt_maps[1][brain2d])
    fig, axes = plt.subplots(2, n, figsize=(3.1 * n, 7.0), squeeze=False)
    for j in range(n):
        axes[0, j].imshow(np.rot90(brain_only(fa_imgs[j], brain2d)),
                          cmap=FA_CM, vmin=0.0, vmax=1.0)
        axes[0, j].set_title(labels[j], fontsize=13, color="black")
        axes[0, j].axis("off")
        axes[1, j].imshow(np.rot90(brain_only(adc_imgs[j], brain2d)),
                          cmap=ADC_CM, vmin=adc_lim[0], vmax=adc_lim[1])
        axes[1, j].axis("off")
    fig.text(0.008, 0.72, "FA", rotation=90, va="center", fontsize=16,
             weight="bold", color="black")
    fig.text(0.008, 0.27, "ADC", rotation=90, va="center", fontsize=16,
             weight="bold", color="black")
    fig.suptitle(f"{subj} - {level} degradation (z={z})", fontsize=15, y=0.99,
                 color="black")
    fig.tight_layout(rect=(0.022, 0, 1, 0.94))
    show_close(fig, out)


metric_rows = []
for subj in TEST_SUBJECTS:
    s = load_subject(subj)
    z = pick_slice(s["dwi"], s["bvals"])
    brain3, brain2 = s["mask"], s["mask"][:, :, z]
    tgt_fa, tgt_adc = fa_adc(s["dti6d"])
    for L in LEVELS:
        # One degraded volume + one kept-direction mask, shared by every method.
        dropped = mask_directions(s["bvals"], L["keep_dirs"], seed=SEED)
        kept = ~dropped
        deg = degrade(s["dwi"], L["keep_fraction"], L["noise"], dropped, seed=SEED)
        print(f"{subj} | {L['name']}: reconstructing "
              f"{', '.join(k for k, _l, _c in ACTIVE_METHODS)} ...")
        method_maps = {}
        for key, _label, _color in ACTIVE_METHODS:
            dti6 = recon_method(key, subj, deg, s["bvals"], s["bvecs"], kept, brain3)
            fa3, adc3 = fa_adc(dti6)
            method_maps[key] = (fa3[:, :, z], adc3[:, :, z])
            metric_rows.append({
                "subject": subj, "level": L["name"], "method": key,
                "keep_fraction": L["keep_fraction"], "noise": L["noise"],
                "keep_dirs": L["keep_dirs"],
                "fa_rmse": scalar_map_metrics(tgt_fa, fa3, mask=brain3)["rmse"],
                "adc_rmse": scalar_map_metrics(tgt_adc, adc3, mask=brain3)["rmse"],
            })
        render_methods(subj, L["name"], z, brain2,
                       (tgt_fa[:, :, z], tgt_adc[:, :, z]), method_maps,
                       FIG_DIR / f"fig_pred_{subj}_{L['name']}.png")

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(REPO / "paper_metrics.csv", index=False)
print("wrote", REPO / "paper_metrics.csv")
metrics_df


## Part 2 — Summary: reconstruction error vs degradation level

In [ ]:
# ── Summary line plot (mean over test subjects, every method) ────────────────
order = [L["name"] for L in LEVELS]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, metric, ttl in zip(axes, ("fa_rmse", "adc_rmse"), ("FA RMSE", "ADC RMSE")):
    for key, label, color in ACTIVE_METHODS:
        g = (metrics_df[metrics_df.method == key]
             .groupby("level")[metric].mean().reindex(order))
        marker = "s-" if key == "ai" else "o--"
        ax.plot(order, g.values, marker, color=color, markersize=8, linewidth=2,
                label=label)
    ax.set_title(ttl, fontsize=14)
    ax.set_xlabel("degradation level")
    ax.set_ylabel(metric)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=11)
fig.suptitle("Reconstruction error vs degradation level (mean over test subjects)",
             fontsize=15)
fig.tight_layout()
show_close(fig, FIG_DIR / "fig_metrics_vs_degradation.png")
metrics_df.pivot_table(index="level", columns="method",
                       values=["fa_rmse", "adc_rmse"]).reindex(order)


## Part 3 — Per-voxel agreement (density scatter + Bland-Altman)

The maps in Part 2 show *where* each method agrees with the target. These plots
show *how well* it agrees across the whole brain, pooled over all test-subject
voxels at a single representative degradation level (`AGREE_LEVEL`, default
"moderate"). Every reconstruction method (Naive, BM4D, Patch2Self, MP-PCA, AI)
gets its own column.

- **Density scatter** — predicted vs target FA/ADC as a log-density hexbin with
  the identity line. A perfect method lies on the diagonal; the inset reports
  brain-masked R² and RMSE.
- **Bland-Altman** — agreement plot of difference (estimate − target) against
  their mean, with the mean bias and ±1.96 SD limits of agreement. Reveals
  systematic bias and how it varies across the FA/ADC range, which a single RMSE
  number hides.

Voxels are restricted to the DIPY brain mask.

In [ ]:
# ── Part 3: gather per-voxel FA/ADC over test subjects (every method) ────────
# Single representative level keeps the scatter readable; change to taste.
AGREE_LEVEL = next(L for L in LEVELS if L["name"] == "moderate")


def gather_voxels(level):
    """Pool brain-masked (target + every method) FA/ADC voxels over all test
    subjects at one degradation level. Reuses the Part-2 degradation +
    reconstruction path so all methods see identical inputs."""
    cols = {"target_fa": [], "target_adc": []}
    for key, _l, _c in ACTIVE_METHODS:
        cols[f"{key}_fa"] = []
        cols[f"{key}_adc"] = []
    for subj in TEST_SUBJECTS:
        s = load_subject(subj)
        m = s["mask"]
        tgt_fa, tgt_adc = fa_adc(s["dti6d"])
        dropped = mask_directions(s["bvals"], level["keep_dirs"], seed=SEED)
        kept = ~dropped
        deg = degrade(s["dwi"], level["keep_fraction"], level["noise"], dropped, seed=SEED)
        cols["target_fa"].append(tgt_fa[m])
        cols["target_adc"].append(tgt_adc[m])
        for key, _l, _c in ACTIVE_METHODS:
            dti6 = recon_method(key, subj, deg, s["bvals"], s["bvecs"], kept, m)
            fa, adc = fa_adc(dti6)
            cols[f"{key}_fa"].append(fa[m])
            cols[f"{key}_adc"].append(adc[m])
    return {k: np.concatenate(v) for k, v in cols.items()}


vox = gather_voxels(AGREE_LEVEL)
ADC_LIM = _robust_limits(vox["target_adc"])
print(f"gathered {vox['target_fa'].size:,} brain voxels at "
      f"'{AGREE_LEVEL['name']}' level (keep={AGREE_LEVEL['keep_fraction']}, "
      f"noise={AGREE_LEVEL['noise']}, keep_dirs={AGREE_LEVEL['keep_dirs']})")


In [ ]:
# ── Part 3a: predicted-vs-target density scatter (every method) ──────────────
def density_scatter(ax, ref, est, lim, title, gridsize=70):
    hb = ax.hexbin(ref, est, gridsize=gridsize, bins="log", cmap="viridis",
                   extent=(lim[0], lim[1], lim[0], lim[1]), mincnt=1)
    ax.plot(lim, lim, "--", color="black", lw=1.3, alpha=0.8)
    ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal")
    r2 = float(np.corrcoef(ref, est)[0, 1] ** 2)
    rmse = float(np.sqrt(np.mean((ref - est) ** 2)))
    ax.text(0.04, 0.96, f"$R^2$ = {r2:.3f}\nRMSE = {rmse:.3g}",
            transform=ax.transAxes, va="top", ha="left", fontsize=11,
            color="black",
            bbox=dict(boxstyle="round", fc="white", ec="gray", alpha=0.85))
    ax.set_title(title, fontsize=12, color="black")
    return hb


metrics_spec = [("fa", (0.0, 1.0), "FA"), ("adc", ADC_LIM, "ADC")]
ncols = len(ACTIVE_METHODS)
fig, axes = plt.subplots(2, ncols, figsize=(4.0 * ncols, 8.6), squeeze=False)
hb = None
for r, (metric, lim, mlabel) in enumerate(metrics_spec):
    for c, (key, label, _color) in enumerate(ACTIVE_METHODS):
        ax = axes[r, c]
        hb = density_scatter(ax, vox[f"target_{metric}"], vox[f"{key}_{metric}"],
                             lim, f"{mlabel} — {label}")
        if r == 1:
            ax.set_xlabel("target", color="black")
        if c == 0:
            ax.set_ylabel("estimate", color="black")
cb = fig.colorbar(hb, ax=axes, fraction=0.025, pad=0.02)
cb.set_label("voxel count (log)", fontsize=11, color="black")
fig.suptitle(f"Predicted vs target ({AGREE_LEVEL['name']} degradation, "
             f"{len(TEST_SUBJECTS)} test subjects)", fontsize=15)
show_close(fig, FIG_DIR / "fig_density_scatter.png")


In [ ]:
# ── Part 3b: Bland-Altman agreement (every method) ───────────────────────────
def bland_altman(ax, ref, est, title, gridsize=70):
    mean = 0.5 * (ref + est)
    diff = est - ref                         # estimate - target (bias > 0 = overestimate)
    md, sd = float(diff.mean()), float(diff.std())
    lo, hi = md - 1.96 * sd, md + 1.96 * sd
    hb = ax.hexbin(mean, diff, gridsize=gridsize, bins="log", cmap="Greys",
                   mincnt=1)
    ax.axhline(0.0, color="black", lw=0.8, alpha=0.4)
    ax.axhline(md, color="#2196F3", lw=1.6, label=f"bias {md:+.3g}")
    ax.axhline(hi, color="#D32F2F", ls="--", lw=1.3,
               label=f"±1.96 SD ({sd:.3g})")
    ax.axhline(lo, color="#D32F2F", ls="--", lw=1.3)
    ax.set_title(title, fontsize=12, color="black")
    ax.set_xlabel("mean of target & estimate", color="black")
    ax.set_ylabel("estimate - target", color="black")
    ax.legend(fontsize=9, loc="upper right", framealpha=0.85)
    return hb


metrics_spec = [("fa", "FA"), ("adc", "ADC")]
ncols = len(ACTIVE_METHODS)
fig, axes = plt.subplots(2, ncols, figsize=(4.2 * ncols, 8.6), squeeze=False)
for r, (metric, mlabel) in enumerate(metrics_spec):
    for c, (key, label, _color) in enumerate(ACTIVE_METHODS):
        bland_altman(axes[r, c], vox[f"target_{metric}"], vox[f"{key}_{metric}"],
                     f"{mlabel} — {label}")
fig.suptitle(f"Bland-Altman agreement ({AGREE_LEVEL['name']} degradation, "
             f"{len(TEST_SUBJECTS)} test subjects)", fontsize=15)
fig.tight_layout()
show_close(fig, FIG_DIR / "fig_bland_altman.png")
